# ResearchPilot — Local PDF Question Answering with a Hugging Face LLM

**ResearchPilot** is a multi-agent research and decision-intelligence system. Given a question, it plans research, gathers evidence from the web and from the user's own documents, checks that claims are supported, and produces a source-backed answer.

This notebook demonstrates the **document question-answering pipeline** end to end, running a real language model locally (no paid API, no API key):

1. Read a PDF page by page.
2. Split it into overlapping chunks that remember their page number.
3. Turn each chunk into an embedding and index it with FAISS.
4. For a question, retrieve the most relevant chunks.
5. Ask a local LLM (`Qwen/Qwen2.5-1.5B-Instruct`) to answer **using only those chunks**.
6. Check the model's citations in code and map them back to real pages, or abstain when the PDF does not contain the answer.

The notebook does **not** call the ResearchPilot API server. It imports the same Python modules the server uses (`app/rag/*`, `app/services/llm.py`), so what you see here is the production code path, not a separate demo implementation.

## 1. Project overview

**The problem.** A language model on its own answers from what it memorised during training. It cannot see your PDF, it may be out of date, and it can produce fluent but invented statements ("hallucinations") with no way to check them. For research and decision support we need answers that are traceable to a specific page of a specific document.

**The approach: retrieval-augmented generation (RAG).** Instead of asking the model to remember, we *retrieve* the relevant passages from the document and put them in the prompt. The model's job becomes reading comprehension over evidence we chose, and every answer can be traced back to those passages.

### Architecture of the pipeline in this notebook

```mermaid
flowchart LR
    PDF[PDF file] --> X[Extract text per page<br/>app/rag/pdf.py]
    X --> C[Chunk with overlap<br/>keeps page numbers<br/>app/rag/chunking.py]
    C --> E[Embedding model<br/>all-MiniLM-L6-v2<br/>384-dim vectors]
    E --> I[(FAISS index<br/>cosine similarity)]
    Q[Question] --> QE[Embed question]
    QE --> I
    I -->|top-k chunks| P[Prompt with evidence<br/>E1, E2, E3]
    P --> L[Local LLM<br/>Qwen2.5-1.5B-Instruct]
    L --> V[Validate cited IDs and<br/>check wording support<br/>app/rag/qa.py]
    V --> A[Answer + real page references<br/>or abstain]
```

(If Mermaid does not render in your viewer, read it left to right: PDF → pages → chunks → embeddings → FAISS; question → embedding → top chunks → prompt → LLM → citation validation → answer.)

### LLM vs. embedding model

| | Embedding model (`all-MiniLM-L6-v2`) | Language model (`Qwen2.5-1.5B-Instruct`) |
|---|---|---|
| Input → output | text → a fixed-length vector of 384 numbers | text (prompt) → new text, one token at a time |
| Purpose here | measure how similar a question is to each chunk | read the retrieved evidence and write the answer |
| Size | about 22 million parameters | about 1.5 billion parameters |
| Runs | once per chunk at indexing time, once per question | once per question |

The embedding model **finds** evidence; the LLM **reads** it. Neither is trained or fine-tuned in this notebook; both are used for inference only.

### Why RAG instead of just asking the LLM?

- **Grounding:** the model sees the actual text, so it does not have to rely on memory.
- **Traceability:** every passage in the prompt has a known filename, page, and chunk id, so citations can be verified by code.
- **Updatability:** adding a document only means indexing it; the model is unchanged.
- **Context limits:** a whole PDF may not fit in the prompt; retrieval selects the few passages that matter.

## 2. Environment setup

**Kaggle settings before running:**

1. **Internet:** open the right-hand panel → *Settings* → *Internet* → **On**. This is needed to `pip install` missing packages and to download the models from Hugging Face (about 90 MB for the embedding model and about 3.1 GB for the LLM). If Internet must stay off, attach the packages and model files as Kaggle inputs instead and point `LLM_MODEL_ID` / `EMBEDDING_MODEL_ID` at their local folders.
2. **Accelerator (optional):** *Settings* → *Accelerator* → **GPU T4 x2** is fastest. The notebook also runs on CPU, only slower (see the hardware notes in section 6).
3. **Inputs:** attach the ResearchPilot code as a Kaggle Dataset (see below) and, optionally, your own PDF.

**Attaching the ResearchPilot code.** On your computer, from the repository root, create a zip of the shared modules and sample data:

```bash
zip -r researchpilot-code.zip app notebooks/sample_data -x '*/__pycache__/*'
```

On Kaggle: *Create* → *New Dataset* → upload `researchpilot-code.zip` → *Create*. In this notebook: *Add Input* → *Your Datasets* → select it. Kaggle extracts the zip under `/kaggle/input/`. The next cell finds the code automatically. (Alternatively, if the repository is on GitHub, set `RESEARCHPILOT_REPO_URL` and the cell will `git clone` it into `/kaggle/working`.)

The configuration cell below is the only place you need to edit.

In [ ]:
# ---- Configuration: edit here ---------------------------------------------------------
import os

PDF_PATH = os.environ.get("PDF_PATH", "")  # leave empty to auto-detect a PDF under /kaggle/input
LLM_MODEL_ID = os.environ.get("LLM_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")
EMBEDDING_MODEL_ID = os.environ.get("EMBEDDING_MODEL_ID", "sentence-transformers/all-MiniLM-L6-v2")
DEVICE = os.environ.get("LLM_DEVICE", "auto")  # "auto", "cuda", "cpu" (or "mps" on a Mac)

CHUNK_SIZE = 500  # characters per chunk
CHUNK_OVERLAP = 100  # characters shared by consecutive chunks
TOP_K = 3  # chunks retrieved per question
MIN_SCORE = 0.25  # cosine similarity below which a chunk is treated as irrelevant
MAX_NEW_TOKENS = 256  # upper bound on the length of each generated answer

REPO_URL = os.environ.get("RESEARCHPILOT_REPO_URL", "")  # optional: git clone source

In [ ]:
# ---- Locate the shared ResearchPilot modules -------------------------------------------
import glob
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path | None:
    # 1) running from a local checkout (e.g. notebooks/ inside the repo)
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "app" / "rag" / "qa.py").exists():
            return parent
    # 2) attached as a Kaggle Dataset, or previously cloned into /kaggle/working
    for pattern in ("/kaggle/input/**/app/rag/qa.py", "/kaggle/working/**/app/rag/qa.py"):
        matches = sorted(glob.glob(pattern, recursive=True))
        if matches:
            return Path(matches[0]).parents[2]
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and REPO_URL:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/researchpilot"], check=True
    )
    PROJECT_ROOT = Path("/kaggle/working/researchpilot")
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "ResearchPilot code not found. Attach the 'researchpilot-code' Kaggle Dataset "
        "(see the instructions above) or set RESEARCHPILOT_REPO_URL."
    )
sys.path.insert(0, str(PROJECT_ROOT))
print("ResearchPilot code:", PROJECT_ROOT)

In [ ]:
# ---- Install only what is missing ------------------------------------------------------
# Kaggle images usually include torch, transformers and sentence-transformers already;
# pypdf and faiss-cpu are typically the only packages that need installing.
import importlib.util

REQUIRED = {  # import name -> pip package
    "pydantic": "pydantic>=2",
    "numpy": "numpy",
    "pypdf": "pypdf",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
    "transformers": "transformers",
    "accelerate": "accelerate",
}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages are already installed.")

In [ ]:
# ---- Python version and hardware -------------------------------------------------------
import platform
from importlib.metadata import version

import torch

from app.services.llm import select_device

assert sys.version_info >= (3, 11), "ResearchPilot needs Python 3.11 or newer"
print(f"{'Python':<23}{platform.python_version()}")
for package in ("torch", "transformers", "sentence-transformers", "faiss-cpu", "pypdf"):
    try:
        print(f"{package:<23}{version(package)}")
    except Exception:
        print(f"{package:<23}(not found)")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

device_config = select_device(DEVICE)
print(f"\nThe LLM will run on: {device_config.device} ({device_config.dtype})")
print(f"Reason: {device_config.reason}")

**Keeping secrets out of the notebook.** This notebook needs no API keys: both models are public. If you want an authenticated Hugging Face download (higher rate limits), add a Kaggle secret named `HF_TOKEN` (*Add-ons* → *Secrets*) and run the optional cell below. It reads the token into an environment variable without printing it. Never paste tokens, Supabase passwords, or service-role keys into a cell, and clear outputs before sharing if a cell ever printed one.

In [ ]:
# Optional: load HF_TOKEN from Kaggle Secrets without displaying it.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets (value hidden).")
except Exception:
    print("No HF_TOKEN secret configured; continuing with anonymous downloads.")

## 3. Load a PDF

**Attaching your own PDF on Kaggle:** *Add Input* → *Upload* (or *New Dataset*) → add the PDF → it appears under `/kaggle/input/<dataset-name>/`. Then either set `PDF_PATH` in the configuration cell, or leave it empty and the cell below picks the first PDF it finds under `/kaggle/input`.

If no PDF is attached, the notebook uses the bundled **synthetic sample report** (`notebooks/sample_data/researchpilot_sample_report.pdf`): a 4-page technical summary about a fictional campus microgrid. Its facts are invented on purpose: if the model answers correctly, it must have read the document rather than recalled training data. Section 8's evaluation questions are written for this sample.

Only PDFs with a text layer are supported. Scanned pages are images; reading them needs OCR (optical character recognition), which this project does not implement, so such pages are reported and skipped.

In [ ]:
from app.rag.pdf import DocumentError, load_pdf

SAMPLE_PDF = PROJECT_ROOT / "notebooks" / "sample_data" / "researchpilot_sample_report.pdf"


def choose_pdf() -> Path:
    if PDF_PATH:
        return Path(PDF_PATH)
    attached = [
        Path(p)
        for p in sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
        if Path(p).resolve() != SAMPLE_PDF.resolve() and "sample_data" not in p
    ]
    return attached[0] if attached else SAMPLE_PDF


pdf_path = choose_pdf()
if not pdf_path.is_file():
    raise FileNotFoundError(
        f"PDF not found: {pdf_path}\n"
        "Attach a PDF via 'Add Input' and set PDF_PATH to its path under /kaggle/input/, "
        "or leave PDF_PATH empty to use the bundled sample report."
    )

try:
    document = load_pdf(pdf_path)
except DocumentError as exc:
    raise RuntimeError(f"Cannot use this PDF: {exc}") from None

USING_SAMPLE = pdf_path.resolve() == SAMPLE_PDF.resolve()
print(f"File:        {document.filename}{'  (bundled synthetic sample)' if USING_SAMPLE else ''}")
print(f"Document id: {document.document_id}   (derived from the file's SHA-256 hash)")
print(f"Pages:       {document.page_count}, with text: {document.pages_with_text}")
print(f"Metadata:    {document.metadata}")
for warning in document.warnings:
    print("WARNING:", warning)

## 4. Extract and chunk the document

Text is extracted **page by page** with `pypdf`, so every piece of text keeps the page it came from. Chunks are then cut **within a page** (never across a page break), which means each chunk maps to exactly one page number.

**Why chunk size and overlap matter:**

- **Too large:** a chunk mixes several topics, its embedding becomes an average of all of them, and a specific question matches it less clearly. It also uses more of the LLM's prompt.
- **Too small:** a chunk loses context (a number without the sentence explaining what it measures).
- **Overlap:** consecutive chunks share some text (here 100 characters), so a sentence cut at a boundary still appears whole in at least one chunk.

Chunk boundaries are moved back to the nearest whitespace, so words are not cut in half. On the sample report, 500-character chunks retrieved more focused passages than 800-character chunks in our tests.

In [ ]:
from IPython.display import Markdown, display

from app.rag.chunking import chunk_document


def md_table(headers, rows):
    def clean(value):
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(clean(v) for v in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


md_table(
    ["Page", "Characters", "First 90 characters"],
    [(p.page_number, len(p.text), p.text[:90] or "(no text layer)") for p in document.pages],
)

chunks = chunk_document(document, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"{len(chunks)} chunks (size {CHUNK_SIZE}, overlap {CHUNK_OVERLAP})")
md_table(
    ["Chunk id", "Page", "Chars", "Text (start)"],
    [(c.chunk_id, c.page_number, len(c.text), c.text[:110] + "...") for c in chunks[:8]],
)

In [ ]:
# Overlap in action: the end of one chunk reappears at the start of the next (same page).
pairs = [(a, b) for a, b in zip(chunks, chunks[1:], strict=False) if a.page_number == b.page_number]
if pairs:
    a, b = pairs[0]
    shared = a.end_char - b.start_char
    print(f"{a.chunk_id} ends at char {a.end_char}; {b.chunk_id} starts at char {b.start_char}")
    print(f"Shared text ({shared} chars): ...{a.text[-shared:]!r}")
else:
    print("Every page fits in a single chunk, so there is no overlap to show.")

## 5. Generate embeddings and build the vector index

`all-MiniLM-L6-v2` maps any text to a 384-dimensional vector such that texts with similar meaning get vectors pointing in similar directions. Vectors are **L2-normalised** (length 1), so the inner product of two vectors equals their **cosine similarity**: 1 means same direction, 0 means unrelated.

The FAISS index used here (`IndexFlatIP`) is a *flat*, exact index: for each question it compares the question vector with every chunk vector and returns the highest inner products. That is exact and fast for thousands of chunks; approximate indexes (HNSW, IVF) only become necessary at much larger scale.

In [ ]:
import time

import numpy as np

from app.rag.embeddings import SentenceTransformerEmbeddings
from app.rag.retrieval import VectorRetriever
from app.rag.vector_store import FaissVectorStore

started = time.perf_counter()
embedder = SentenceTransformerEmbeddings(EMBEDDING_MODEL_ID)
vectors = embedder.embed_documents([c.text for c in chunks])
elapsed = time.perf_counter() - started
print(f"Embedding model: {embedder.model_id} (loaded and encoded in {elapsed:.1f}s)")
print(f"Embedding matrix: shape {vectors.shape}, dtype {vectors.dtype}")
print(f"Vector lengths (should all be 1.0): {np.round(np.linalg.norm(vectors, axis=1)[:5], 4)}")
print(f"First 8 numbers of chunk 0: {np.round(vectors[0][:8], 3)}")

store = FaissVectorStore(embedder.dimension)
store.add(chunks, vectors)
retriever = VectorRetriever(embedder, store, min_score=MIN_SCORE)
print(f"\nFAISS index: {store.index.ntotal} vectors of dimension {store.dimension}")

In [ ]:
# Similarity search: the top chunks for a query, with their scores and real page numbers.
SEARCH_QUERY = "Why was this battery chemistry selected?" if USING_SAMPLE else "main findings"

results = store.search(embedder.embed_query(SEARCH_QUERY), top_k=5)
print(f"Query: {SEARCH_QUERY!r}\n")
md_table(
    ["Rank", "Similarity", "File", "Page", "Chunk id", "Text"],
    [
        (
            r.rank,
            f"{r.score:.3f}",
            r.chunk.filename,
            r.chunk.page_number,
            r.chunk.chunk_id,
            r.chunk.text[:100] + "...",
        )
        for r in results
    ],
)
print(f"Chunks scoring below MIN_SCORE={MIN_SCORE} are dropped before the LLM sees them.")

## 6. Load the local LLM

`HuggingFaceLLM` (in `app/services/llm.py`) wraps Hugging Face Transformers:

- **Lazy loading.** Creating the object does nothing; the model downloads and loads on `load()` or the first `generate()`. This is why importing the ResearchPilot API never pulls a 3 GB model.
- **Hardware selection** (`select_device`): a CUDA GPU if PyTorch supports it, otherwise CPU (Apple GPUs via MPS locally). If Kaggle assigns a GPU that the installed PyTorch build was not compiled for (this has happened with older P100 GPUs), the notebook falls back to CPU with an explanation instead of crashing.
- **Precision (dtype).** On GPU the weights are loaded in 16-bit (`float16` on a T4, `bfloat16` on Ampere or newer), about **3.1 GB** for 1.54 billion parameters. On CPU they are loaded in `float32` (about **6.2 GB of RAM**), because 16-bit maths on CPU is slow or unsupported for many operations.
- **CPU vs GPU trade-off.** A GPU is much faster at the matrix multiplications that dominate inference; a T4 should answer in seconds (not benchmarked for this project; the timings printed in section 7 are the real numbers for your session). A CPU gives essentially the same answers, since greedy decoding is deterministic, though 16-bit and 32-bit arithmetic can differ in rare tie-breaks, but more slowly: during development, an Apple M4 Pro CPU generated about 20 tokens per second in float32, and Kaggle's 4-core CPU sessions will be slower. Kaggle CPU sessions typically have about 30 GB of RAM, enough for float32.
- **If loading fails** (no Internet, wrong model id, out of memory), `LLMLoadError` explains what to change. There is no fallback to fake output.

**Why Qwen2.5-1.5B-Instruct?** It is instruction-tuned (trained to follow a system prompt and a requested format), small enough to fit a free Kaggle T4 in 16-bit and to run on CPU, has a 32K-token context window, and is released under the Apache-2.0 licence. Larger models follow instructions more reliably but would not fit the free hardware comfortably; set `LLM_MODEL_ID` to try another chat model, e.g. `Qwen/Qwen2.5-3B-Instruct` on a GPU.

In [ ]:
from app.services.llm import HuggingFaceLLM, LLMLoadError

llm = HuggingFaceLLM(LLM_MODEL_ID, device=DEVICE)
print("Before load():", llm.describe())
started = time.perf_counter()
try:
    llm.load()
except LLMLoadError as exc:
    print(exc)
    raise
print(f"Loaded in {time.perf_counter() - started:.1f}s")
for key, value in llm.describe().items():
    print(f"  {key:<20}{value}")

### Tokenization, prompt construction, and inference

1. **Tokenization.** The model does not read characters; the tokenizer splits text into sub-word *tokens* and maps each to an integer id from a vocabulary of about 150,000 entries.
2. **Prompt construction.** ResearchPilot builds a list of chat messages: a *system* message with the rules (use only the passages; reply as `ANSWER:` and `EVIDENCE:`; reply `INSUFFICIENT_EVIDENCE` if the passages do not answer), and a *user* message containing only the retrieved passages, labelled `Passage E1`, `Passage E2`, ..., plus the question. The tokenizer's **chat template** wraps these messages in the special role markers the model was trained on.
3. **Inference.** The model predicts the next token from all previous tokens, appends it, and repeats until it emits an end-of-turn token or reaches `MAX_NEW_TOKENS`. We use **greedy decoding** (always pick the most likely token), so the same prompt always gives the same answer, which makes results reproducible and evaluation meaningful.

In [ ]:
sample = "The microgrid uses lithium iron phosphate batteries."
token_ids = llm.tokenizer(sample)["input_ids"]
print(f"{len(token_ids)} tokens: {llm.tokenizer.convert_ids_to_tokens(token_ids)}")
print(f"ids: {token_ids}")

In [ ]:
from app.rag.qa import build_prompt

demo_question = (
    "What is the peak capacity of the solar array?"
    if USING_SAMPLE
    else "What is this document about?"
)
demo_messages = build_prompt(demo_question, retriever.retrieve(demo_question, top_k=2))
for message in demo_messages:
    print(f"--- {message['role']} ---\n{message['content']}\n")

templated = llm.tokenizer.apply_chat_template(
    demo_messages, tokenize=False, add_generation_prompt=True
)
print("--- exactly what the model receives (after the chat template), last 400 characters ---")
print(templated[-400:])

## 7. Evidence-grounded question answering

`answer_question()` (in `app/rag/qa.py`) runs the full pipeline:

1. **Retrieve** the top `TOP_K` chunks whose similarity is at least `MIN_SCORE`. If none qualifies, it abstains immediately **without calling the LLM**.
2. **Construct the prompt** from those chunks only, labelled `E1..Ek`.
3. **Generate** with the local LLM.
4. **Parse** the `ANSWER:` text and the passage IDs on the `EVIDENCE:` line.
5. **Resolve** each ID to the chunk that was actually retrieved for this question; the filename and page number come from that chunk's stored metadata, never from text the model wrote.
6. **Validate**: IDs that were not in the prompt are discarded and reported. Each answer sentence is then checked against the cited passages: if fewer than half of its content words appear in them, it is flagged as weakly supported. Each cited passage is also checked on its own: if it contains less than 30% of the answer's content words, it is still shown but marked as *not supporting* the answer (small models tend to over-cite).
7. **Decide**: `answered`; `insufficient_evidence` (no relevant chunk, or the model abstained); or `unsupported` (no valid citation, or no sentence supported by the cited text), in which case the answer is withheld and only the raw output is kept for inspection.

Change `QUESTION` and re-run the cell to ask your own question.

In [ ]:
from app.rag.qa import answer_question, render_markdown


def ask(question: str, show_evidence: bool = True):
    result = answer_question(question, retriever, llm, top_k=TOP_K, max_new_tokens=MAX_NEW_TOKENS)
    display(Markdown(render_markdown(result)))
    if show_evidence and result.retrieved:
        md_table(
            ["ID", "Chunk id", "Page", "Similarity", "Passage given to the model"],
            [
                (
                    label,
                    r.chunk.chunk_id,
                    r.chunk.page_number,
                    f"{r.score:.3f}",
                    r.chunk.text[:120] + "...",
                )
                for label, r in zip(result.evidence_labels, result.retrieved, strict=True)
            ],
        )
    print("Raw model output:", repr(result.raw_output))
    print("Invalid (discarded) IDs:", result.invalid_citations or "none")
    for c in result.citations:
        verdict = "supports" if c.supports_answer else "does NOT support"
        print(
            f"  {c.label} (page {c.page_number}) {verdict} the answer: {c.overlap:.0%} word overlap"
        )
    for s in result.sentence_support:
        print(f"  support {s.overlap:.0%} {'OK  ' if s.supported else 'WEAK'} {s.sentence}")
    m = result.metrics
    print(
        f"Timing: retrieval {m.retrieval_ms:.0f} ms, generation {m.generation_ms:.0f} ms; "
        f"tokens: {m.prompt_tokens} prompt + {m.completion_tokens} generated"
    )
    return result


QUESTION = (
    "Which battery chemistry did the microgrid pilot use, and why was it chosen?"
    if USING_SAMPLE
    else "What is the main conclusion of this document?"
)
result = ask(QUESTION)

In [ ]:
# The structured result (what the API or a later agent would consume):
print(
    result.model_dump_json(
        indent=2,
        include={
            "question",
            "status",
            "answer",
            "citations",
            "invalid_citations",
            "evidence_support",
            "evidence_labels",
            "warnings",
        },
    )
)

### The citation validator, tested in isolation

The next cell replaces the LLM with a **scripted stub** that returns hand-written replies, to show what the validator does with fabricated IDs, missing citations, and cited-but-unrelated text. **This cell involves no AI**; it tests ResearchPilot's checking code, which is what keeps a real model's mistakes from reaching the user.

In [ ]:
from app.services.llm import Generation


class ScriptedReply:
    model_id = "scripted-stub (not a model)"

    def __init__(self, text):
        self.text = text

    def generate(self, messages, *, max_new_tokens=256):
        return Generation(self.text, prompt_tokens=0, completion_tokens=0, duration_ms=0.0)


q = demo_question
cases = {
    "fabricated ID E9": "ANSWER: The array has a peak capacity of 1.2 megawatts.\nEVIDENCE: E1, E9",
    "no citation": "ANSWER: The array has a peak capacity of 1.2 megawatts.\nEVIDENCE: none",
    "cited but unrelated": "ANSWER: The containers are guarded by robots.\nEVIDENCE: E1",
}
for name, reply in cases.items():
    r = answer_question(q, retriever, ScriptedReply(reply), top_k=TOP_K)
    print(
        f"{name:<22} status={r.status.value:<22} valid={[c.label for c in r.citations]} "
        f"discarded={r.invalid_citations}"
    )

if USING_SAMPLE:
    r = answer_question(q, retriever, ScriptedReply(cases["fabricated ID E9"]), top_k=TOP_K)
    assert r.invalid_citations == ["E9"] and all(c.label != "E9" for c in r.citations)
    assert (
        answer_question(q, retriever, ScriptedReply(cases["no citation"])).status.value
        == "unsupported"
    )
    assert (
        answer_question(q, retriever, ScriptedReply(cases["cited but unrelated"])).status.value
        == "unsupported"
    )
    print("\nValidator checks passed.")

## 8. Demonstration and evaluation

Two contrasting questions first: one the document answers, one it does not. The second should end in `insufficient_evidence`, either because no chunk is similar enough or because the model reads the passages and abstains.

In [ ]:
ANSWERABLE = (
    "How did the microgrid perform during grid outages?"
    if USING_SAMPLE
    else "What problem does this document address?"
)
UNANSWERABLE = (
    "Who is the president of Lakeside Technical University?"
    if USING_SAMPLE
    else "What was the share price of Apple on 1 March 1999?"
)

print("=" * 30, "answerable", "=" * 30)
answerable_result = ask(ANSWERABLE)
print("\n" + "=" * 30, "not in the document", "=" * 30)
unanswerable_result = ask(UNANSWERABLE)

### Evaluation suite

Each case states whether the document answers the question and on which page the evidence is. For the sample report the cases come from `notebooks/sample_data/sample_eval_cases.json`. For your own PDF, edit `CUSTOM_CASES` below.

Metrics (all computed from the runs in this cell, nothing estimated):

| Metric | Meaning |
|---|---|
| `retrieval_hit_rate` | answerable cases where an expected page is among the retrieved chunks |
| `decision_accuracy` | answered when answerable, abstained when not |
| `cited_expected_page_rate` | answered cases whose *supporting* citations include an expected page |
| `citation_validity` | cited IDs that refer to retrieved passages, over all cited IDs |
| `mean_evidence_support` | average share of answer sentences supported by the cited text |

In [ ]:
import json

from app.rag.evaluation import EvalCase, evaluate_qa, evaluate_retrieval, summarize

CUSTOM_CASES = [
    # {"id": "my-question", "question": "...", "answerable": True, "expected_pages": [3]},
]

if USING_SAMPLE:
    raw_cases = json.loads(
        (PROJECT_ROOT / "notebooks/sample_data/sample_eval_cases.json").read_text()
    )
else:
    raw_cases = CUSTOM_CASES
eval_cases = [EvalCase(**c) for c in raw_cases]

if not eval_cases:
    print("No evaluation cases for this PDF yet: add some to CUSTOM_CASES and re-run.")
else:
    full_results = {}  # question -> QAResult, reused by the checks in the next cell

    def run_case(question):
        full_results[question] = answer_question(
            question, retriever, llm, top_k=TOP_K, max_new_tokens=MAX_NEW_TOKENS
        )
        return full_results[question]

    retrieval_results = evaluate_retrieval(retriever, eval_cases, top_k=TOP_K)
    qa_results = evaluate_qa(run_case, eval_cases)
    md_table(
        [
            "Case",
            "Answerable",
            "Expected pages",
            "Retrieved pages",
            "Status",
            "Supporting cited pages",
            "Correct decision",
            "Support",
            "Latency (ms)",
        ],
        [
            (
                c.id,
                c.answerable,
                c.expected_pages,
                r.retrieved_pages,
                q.status.value,
                q.cited_pages,
                q.correct_decision,
                "-" if q.evidence_support is None else f"{q.evidence_support:.0%}",
                f"{q.latency_ms:.0f}",
            )
            for c, r, q in zip(eval_cases, retrieval_results, qa_results, strict=True)
        ],
    )
    summary = summarize(retrieval_results, qa_results)
    print(json.dumps(summary, indent=2))

In [ ]:
# Automated checks on the runs above. The first group must hold whatever the model writes,
# because code enforces it: every citation points to a chunk retrieved for that question, with
# that chunk's real page. The last check tests the model's behaviour: it must not answer the
# questions marked unanswerable. If it fails, that is a genuine finding to report.
if eval_cases:
    for case in eval_cases:
        r = full_results[case.question]
        retrieved_ids = {item.chunk.chunk_id for item in r.retrieved}
        for c in r.citations:
            assert c.chunk_id in retrieved_ids, f"{case.id}: citation to an unretrieved chunk"
            chunk = next(i.chunk for i in r.retrieved if i.chunk.chunk_id == c.chunk_id)
            assert (c.page_number, c.filename) == (chunk.page_number, chunk.filename)
        assert all(item.score >= MIN_SCORE for item in r.retrieved), "threshold not applied"
        assert [item.rank for item in r.retrieved] == list(range(1, len(r.retrieved) + 1))
        if not case.answerable:
            assert r.status.value != "answered", f"{case.id}: answered an unanswerable question"
    print(f"Citation-validity and retrieval checks passed for {len(eval_cases)} cases.")

### How to read these results, and the limitations

- **Small, tuned test set.** The sample has 7 questions, and the prompt format was refined while testing on these same questions, so the scores above are an optimistic sanity check, not a benchmark. A credible evaluation needs more questions, written by someone who did not tune the prompt, on documents the system has not seen.
- **Plausible vs. verified.** A fluent answer is only *plausible*. ResearchPilot treats it as verified only to the extent code can check it: the cited passages were really retrieved, and the sentence's wording appears in them. The wording check is lexical; it cannot detect a sentence that reuses the passage's words but reverses its meaning (e.g. adding "not"). Read the cited page before relying on an answer.
- **Hallucination risk.** The model can still add a detail that is not in the passages. The prompt forbids it and the support check catches many cases, but not all. The raw output is always kept so a person can inspect it.
- **Retrieval mistakes.** If the right chunk is not in the top `TOP_K`, the model never sees it and should abstain, which looks like "the document does not say" even though it does. Similarity scores also do not separate answerable from unanswerable questions cleanly (in testing, an unanswerable question about the university president scored about 0.5 against page 1), which is why abstention relies on the model reading the evidence, not on the threshold alone.
- **Context-window limits.** Qwen2.5 accepts 32K tokens, but only `TOP_K` short chunks are sent, to keep generation fast and focused. Questions that need facts spread across many pages may be answered incompletely.
- **Over-citation.** The model sometimes lists a passage it did not use (in testing, a page-4 passage alongside the correct page-2 passage). The per-citation overlap check catches clear cases, but the 30% threshold was chosen from a handful of examples.
- **Model size.** A 1.5B-parameter model sometimes ignores the requested format (for example, by leaving out the EVIDENCE line). ResearchPilot then withholds the answer rather than guessing which passage it used.
- **PDF extraction.** Scanned pages (no OCR), complex tables, and multi-column layouts may extract poorly.
- **Hardware variance.** Results were checked with float32 on CPU; float16 on a GPU can change rare word choices.

## 9. University discussion notes

**Why this model.** `Qwen/Qwen2.5-1.5B-Instruct` is an instruction-tuned, Apache-2.0-licensed chat model with 1.54B parameters and a 32K-token context. In 16-bit it needs about 3.1 GB, so it fits a free Kaggle T4; in float32 it also runs on CPU. It was used as-is, without fine-tuning: the task is reading comprehension over supplied evidence, which an instruction-tuned model can do with a clear prompt.

**What the embedding model does.** `all-MiniLM-L6-v2` (a 6-layer sentence-transformer) converts each chunk and each question into a 384-dimensional unit vector. Texts about the same thing get vectors with high cosine similarity even when they use different words, e.g. "battery chemistry" matching "lithium iron phosphate".

**How RAG works here.** Offline: PDF → page text → overlapping 500-character chunks → embeddings → FAISS index. Per question: embed the question → retrieve the most similar chunks → put only those chunks in the prompt → the LLM writes an answer and lists the passages it used.

**How the vector index retrieves evidence.** `FaissVectorStore` uses `IndexFlatIP`: it computes the inner product (cosine similarity, because vectors are normalised) between the question and every chunk, returns the top `k`, and ResearchPilot drops results below `MIN_SCORE`. The index is exact; for millions of chunks we would switch to an approximate index or the optional Supabase pgvector backend (HNSW).

**How evidence reaches the LLM.** `build_prompt()` creates a system message with the rules and output format and a user message with the passages labelled `E1..Ek` and the question. The tokenizer's chat template converts these into the model's input tokens; greedy decoding produces the reply.

**How answers are tied to real PDF pages.** Every chunk stores `document_id`, `filename`, `page_number`, and a stable `chunk_id` (`<document>:p<page>:c<index>`) from the moment it is created. The model only ever outputs passage IDs such as `E2`; code maps `E2` back to the chunk that was in the prompt and reads the page number from its metadata. IDs not in the prompt are discarded. The model is never trusted to write a page number. Finally, each cited passage must share at least 30% of the answer's content words to count as support; passages the model lists without using are shown separately.

**What happens when evidence is insufficient.** Three safeguards: (1) if no chunk reaches the similarity threshold, the system abstains without calling the LLM; (2) the model is instructed to reply `INSUFFICIENT_EVIDENCE` when the passages do not answer the question; (3) an answer with no valid citation, or whose sentences are not supported by the cited text, is withheld and labelled `unsupported`.

**How this connects to the multi-agent LangGraph workflow.** The same `DocumentLibrary` and retriever power the `document_search` node of ResearchPilot's LangGraph workflow, which runs in parallel with web search for every planned query (`POST /research` with `document_ids`) and labels each source as `document`, `web`, or synthetic `demo`. With `"analyze": true`, the API then runs three more agents on the collected sources: an **Analyst** (this same Qwen model, via the same `HuggingFaceLLM` class) that writes JSON findings citing source IDs; a **Verifier** that checks every citation in code (IDs exist, document chunks and pages match the store, numbers and wording appear in the source, the compared subject is mentioned, no injected instructions) and asks the model to review each claim-source pair, sending rejected claims back for at most one revision; and a deterministic **Report Writer**. This notebook demonstrates the single-question version of that pipeline (`answer_question()`); the API workflow is shown with `scripts/demo_api.py`.

**Limitations and improvements.** Lexical support checking → natural-language-inference (entailment) or LLM-judge verification; flat FAISS → HNSW/pgvector at scale; add OCR for scanned PDFs; hybrid keyword + vector retrieval and a re-ranker for better recall; a larger or quantised model on GPU for more reliable format following; a larger, independently written evaluation set.